# Fungal Annotation with the q2-fungal-traits plugin

### Text what will be done

#### Setup and definition of variables:

In [14]:
import os
import pandas as pd

# ---------- base folders ----------
data_dir = "/home/jovyan/FunGut/data"
ign_dir  = f"{data_dir}/processed_ignored"
vis_dir  = f"{data_dir}/processed"

# ---------- existing inputs ----------
taxonomy_qza = f"{ign_dir}/03_taxonomy/taxonomy.qza"                # taxonomic classification of the ASVs
table_qza    = f"{ign_dir}/03_taxonomy/table_fungi_filtered.qza"    # ASV table, fungi only, NOT rarefied
rep_seqs_qza = f"{ign_dir}/03_taxonomy/rep_set_fungi_filtered.qza"  # ASV sequences
cm_dir       = f"{ign_dir}/06_diversity/core_metrics"               # core-metrics results (unfiltered)
rarefied_qza = f"{cm_dir}/rarefied_table.qza"                       # rarefied ASV table
metadata     = f"{ign_dir}/06_diversity/Final_metadata_with_GHSL.tsv"

# ---------- fungal-traits environment ----------
ft_env   = "/home/jovyan/envs/q2-fungal-traits"
ft_qiime = f"conda run --no-capture-output -p {ft_env} qiime"

# ---------- output folders ----------
base = "Fungal_Annotation"
annot_dir,  annot_vis  = f"{ign_dir}/{base}/01_annotation",           f"{vis_dir}/{base}/01_annotation"
cur_dir,    cur_vis    = f"{ign_dir}/{base}/02_curation",             f"{vis_dir}/{base}/02_curation"
filt_dir,   filt_vis   = f"{ign_dir}/{base}/03_nonintrinsic",         f"{vis_dir}/{base}/03_nonintrinsic"
grp_dir,    grp_vis    = f"{ign_dir}/{base}/04_group_comparisons",    f"{vis_dir}/{base}/04_group_comparisons"
func_dir,   func_vis   = f"{ign_dir}/{base}/05_functional_diversity", f"{vis_dir}/{base}/05_functional_diversity"
ancom_dir,  ancom_vis  = f"{ign_dir}/{base}/06_ancombc",              f"{vis_dir}/{base}/06_ancombc"

for d in [annot_dir, annot_vis, cur_dir, cur_vis, filt_dir, filt_vis,
          grp_dir, grp_vis, func_dir, func_vis, ancom_dir, ancom_vis]:
    os.makedirs(d, exist_ok=True)

# ---------- metadata columns ----------
cat_main = ["diet_type_sample", "gluten_sample", "ibd_sample", "Settlement_Type"]   # main focus, categorical
num_main = ["bmi_sample"]                                                           # main focus, numeric
cat_extra = ["sex_sample", "country_sample"]                                        # optional
num_extra = ["age_years_sample"]                                                    # optional

# ---------- files created during the workflow but will be used frequently ----------
traits_tsv     = f"{annot_dir}/fungal_traits_tsv/metadata.tsv"   # original trait table
traits_curated = f"{cur_dir}/traits_curated.tsv"                  # corrected trait table
exclude_tsv = f"{cur_dir}/exclude_samples_lifestyle.tsv"   # samples excluded from the lifestyle analyses

print("Setup done.")

Setup done.


#### Define helper functions:

In [7]:
import zipfile, tempfile
from biom import load_table

def load_table_qza(path):
    """Feature table (.qza) -> DataFrame: rows = ASVs, columns = samples"""
    with zipfile.ZipFile(path) as z:
        inner = [n for n in z.namelist() if n.endswith("data/feature-table.biom")][0]
        tmp = tempfile.mkdtemp()
        z.extract(inner, tmp)
        return load_table(os.path.join(tmp, inner)).to_dataframe(dense=True)

def load_traits(path):
    """Trait table (.tsv) -> DataFrame with feature-id as index, without QIIME info rows"""
    t = pd.read_csv(path, sep="\t", index_col=0)
    return t[~t.index.astype(str).str.startswith("#")]

def load_alpha(path):
    """Alpha diversity (.qza) -> Series: sample-id -> value"""
    with zipfile.ZipFile(path) as z:
        inner = [n for n in z.namelist() if n.endswith("data/alpha-diversity.tsv")][0]
        return pd.read_csv(z.open(inner), sep="\t", index_col=0).iloc[:, 0]

def load_metadata(path):
    """Sample metadata (.tsv) -> DataFrame with sample-id as index"""
    m = pd.read_csv(path, sep="\t", index_col=0, dtype=str)
    return m[~m.index.astype(str).str.startswith("#")]

print("Helper functions loaded.")

Helper functions loaded.


#### Install fungal-traits environment:

In [5]:
# Create environment:
!conda env create --prefix "$ft_env" --file https://raw.githubusercontent.com/bokulich-lab/q2-fungal-traits/refs/heads/main/environment-files/q2-fungal-traits-rachis-tiny-2026.10.yml
# Install plugin:
!conda run --no-capture-output -p "$ft_env" pip install "q2-fungal-traits@git+https://github.com/bokulich-lab/q2-fungal-traits.git"
# Update QUIIME plugin list:
!$ft_qiime dev refresh-cache
# Check if everything works:
!$ft_qiime fungal-traits annotate --help

Error loading anaconda-anon-usage: module 'conda.cli.install' has no attribute 'check_prefix'
Retrieving notices: done
Channels:
 - https://packages.qiime2.org/qiime2/2026.10/tiny/released
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /home/jovyan/envs/q2-fungal-traits

  added / updated specs:
    - pip
    - rachis-tiny


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    _python_abi3_support-1.0   |       hd8ed1ab_3           8 KB  conda-forge
    _r-mutex-1.0.1             |      anacondar_1           3 KB  conda-forge
    annotated-doc-0.0.5        |     pyhcf101f3_0          16 KB  conda-forge
    array-api-compat-1.16.0    |     pyhc364b38_0          63 KB  conda-forge
    atpublic-9.0.0             |     pyhd8ed1ab_0        

## 01 Annotation

#### Annotate the taxonomy with fungal traits:

In [8]:
!$ft_qiime fungal-traits annotate \
  --i-taxonomy "$taxonomy_qza" \
  --o-fungal-traits "$annot_dir/fungal_traits.qza"

!$ft_qiime tools export \
  --input-path "$annot_dir/fungal_traits.qza" \
  --output-path "$annot_dir/fungal_traits_tsv"

Saved ImmutableMetadata to: /home/jovyan/FunGut/data/processed_ignored/Fungal_Annotation/01_annotation/fungal_traits.qza
Exported /home/jovyan/FunGut/data/processed_ignored/Fungal_Annotation/01_annotation/fungal_traits.qza as ImmutableMetadataDirectoryFormat to directory /home/jovyan/FunGut/data/processed_ignored/Fungal_Annotation/01_annotation/fungal_traits_tsv


In [9]:
!qiime metadata tabulate \
  --m-input-file "$traits_tsv" \
  --o-visualization "$annot_vis/fungal_traits.qzv"

Saved Visualization to: /home/jovyan/FunGut/data/processed/Fungal_Annotation/01_annotation/fungal_traits.qzv


#### Get an overview over the annotation and check if it looks correct in the non-rarefied and rarefied ASV tables. We will further use the rarefied ASV table created during the Alpha Diversity workflow. The depth was set at 1600, based on a alpha rarefaction plot:

In [10]:
traits = load_traits(traits_tsv)
rare   = load_table_qza(rarefied_qza)    # rarefied ASV table (from core-metrics)
raw    = load_table_qza(table_qza)       # non-rarefied ASV table

print("ASVs in trait table:", len(traits))
print("  with lifestyle annotation:", traits["primary_lifestyle"].notna().sum())
print("  without annotation:       ", traits["primary_lifestyle"].isna().sum())
print()

for name, tab in [("Rarefied table", rare), ("Non-rarefied table", raw)]:
    found = tab.index.isin(traits.index)
    print(f"{name}: {tab.shape[0]} ASVs, {tab.shape[1]} samples | "
          f"{found.sum()} ASVs found in trait table "
          f"({tab[found].values.sum() / tab.values.sum() * 100:.1f}% of reads)")
print("Reads per sample (rarefied):", sorted(rare.sum().unique()))

for col in ["fruitbody_type_template", "primary_lifestyle", "growth_form_template"]:
    print(f"\n===== {col}")
    print(traits[col].value_counts(dropna=False))

ASVs in trait table: 681
  with lifestyle annotation: 491
  without annotation:        190

Rarefied table: 547 ASVs, 150 samples | 547 ASVs found in trait table (100.0% of reads)
Non-rarefied table: 575 ASVs, 150 samples | 575 ASVs found in trait table (100.0% of reads)
Reads per sample (rarefied): [np.float64(16000.0)]

===== fruitbody_type_template
fruitbody_type_template
none                                             317
NaN                                              293
perithecium_(hymenium_hidden,_narrow_opening)     35
agaricoid                                         11
cleistothecium_(closed,_spherical)                 7
polyporoid                                         5
corticioid                                         5
apothecium_(hymenium_on_surface)                   4
other                                              2
clavarioid                                         1
cyphelloid                                         1
Name: count, dtype: int64

===== primar

-> We see that the non-rarefied table (= table with reads per ASV per fungi, filtered to fungi) has 575 (from the total initially detected 681) ASVs, some have been removed during 03_taxonomy because they were non-fungi. The rarefied table created in 06_diversity has a bit less ASVs (547) because some very rare ones were lost during rarefying to an equal depth of 1600. In both table 100% of ASVs are found in the trait table, which is what we wanted to check.
#### There are 190 ASVs without "primary lifestyle" annotation, lets look at those:

In [12]:
import numpy as np

unann = traits[traits["primary_lifestyle"].isna()].copy()
unann["genus"] = unann["taxon"].str.extract(r"g__([^;]+)")[0]
unann["category"] = np.select(
    [unann["genus"].isna() | unann["genus"].str.contains("unidentified", na=False),
     unann["genus"].str.contains("Incertae_sedis", na=False)],
    ["no genus", "genus incertae sedis"],
    default="named genus")
unann["in_table"] = unann.index.isin(rare.index)
unann["reads"]    = rare.reindex(unann.index).sum(axis=1)       # reads in the rarefied table (0 if not present)

# 1. overview by category
print(unann.groupby("category").agg(ASVs=("taxon", "size"),
                                    ASVs_in_table=("in_table", "sum"),
                                    reads=("reads", "sum")))

# 2. share of unannotated reads per sample (%)
share_before = rare.loc[rare.index.isin(unann.index)].sum() / rare.sum() * 100
print("\nShare of unannotated reads per sample (%):")
print(share_before.describe().round(2))
print("Samples with >50%:", (share_before > 50).sum())

# 3. named genera = candidates for correction
print("\nNamed genera without annotation:")
print(unann[unann["category"] == "named genus"]
      .groupby("genus").agg(ASVs=("taxon", "size"), reads=("reads", "sum"))
      .sort_values("reads", ascending=False))

# 4. the 10 unannotated ASVs with the most reads, with full taxonomy
pd.set_option("display.max_colwidth", None)
unann.sort_values("reads", ascending=False).head(10)[["reads", "category", "taxon"]]

                      ASVs  ASVs_in_table    reads
category                                          
genus incertae sedis    11             11  13797.0
named genus             17             15  18311.0
no genus               162             52  41335.0

Share of unannotated reads per sample (%):
count    150.00
mean       3.06
std       12.06
min        0.00
25%        0.00
50%        0.00
75%        0.12
max       98.19
dtype: float64
Samples with >50%: 3

Named genera without annotation:
                    ASVs    reads
genus                            
Oligophagozyma         5  18198.0
Podila                 1     32.0
Linnemannia            2     20.0
Pseudopeyronellaea     1     16.0
Candolleomyces         1     12.0
Holleya                1     11.0
Proxiovicillium        2     10.0
Maudiozyma             1      9.0
Alloacremonium         1      2.0
Waltergamsia           1      1.0
Limtongozyma           1      0.0


,reads,category,taxon
feature-id,,,
6f2f7df5a1e2c18abe0b74bdb5a6a5f5,33821.0,no genus,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae
ade5af8c37bb12d50304dfba8581e10e,17077.0,named genus,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Oligophagozyma;s__Oligophagozyma_castellii
3cdc87eb8f6c0721b900fe62f8aa214a,13700.0,genus incertae sedis,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae;g__Dipodascaceae_gen_Incertae_sedis;s__Dipodascaceae_sp
7d75afa6401de7f9c1d9cfbef24ab309,4526.0,no genus,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae
6ffd3dbcfbee803f35e5fe4e5649b1e8,2177.0,no genus,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae
4073a8e805c574783cbf9ae9b5b930c1,1052.0,named genus,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Oligophagozyma;s__Oligophagozyma_castellii
d7300937139cd8e712fa65d74a1eaed4,198.0,no genus,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae
8197f79fb9bdad0f9c1feec25bc66e12,101.0,no genus,k__Fungi;p__Ascomycota;c__Dothideomycetes;o__Pleosporales;f__Didymellaceae
93cb21f5c699c523e48a0168cbfbbda7,64.0,no genus,k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales


-> We can see that 17 of the unannotated ASVs have a classification on genus level (as listed below), but the database that fungal-traits uses just doesnt know them. We can correct those if we find out, what primary lifestyle they have.

-> We see the share of unannotated reads per sample. Most samples have not many (75% quartile is only 0.12) but there are 3 samples with >50% unannotated reads and the maximum is 98.19% unannotated reads in one sample. Maybe this can be improved after annotating the genus level ASVs by hand. Below we can see the 10 unannotated ASVs with the most reads and the top 1 is one with no genus, which cant be corrected

#### Correct "named genus" ASVs by hand where we can/know and assign "yeast" to Dipodascaceae because no matter what genus, all in this family are yeasts:

In [13]:
# (A) Genera described or renamed after FungalTraits v1.2 (Dec 2020): take the former genus's traits.
#     Values looked up in FungalTraits v1.2 (the plugin's database).
genus_map = {
    # genus in data      former genus     primary                  secondary            growth form             fruitbody
    "Oligophagozyma": ("Candida",       "nectar/tap_saprotroph", "",                  "yeast",                "none"),
    "Maudiozyma":     ("Kazachstania",  "nectar/tap_saprotroph", "",                  "yeast",                "none"),
    "Podila":         ("Mortierella",   "soil_saprotroph",       "root-associated",   "filamentous_mycelium", "none"),
    "Linnemannia":    ("Mortierella",   "soil_saprotroph",       "root-associated",   "filamentous_mycelium", "none"),
    "Candolleomyces": ("Psathyrella",   "wood_saprotroph",       "litter_saprotroph", "filamentous_mycelium", "agaricoid"),
}

# (B) Dipodascaceae without a genus: all genera of the family are yeasts -> growth form "yeast".
#     Lifestyle differs within the family (5/6 nectar/tap_saprotroph, Magnusiomyces animal_parasite).
assign_dipodascaceae_lifestyle = False      # Switch: False = leave lifestyle empty, True = nectar/tap_saprotroph

cols = ["primary_lifestyle", "secondary_lifestyle", "growth_form_template", "fruitbody_type_template"]

t = pd.read_csv(traits_tsv, sep="\t", dtype=str, keep_default_na=False)   # ORIGINAL trait table
id_col = t.columns[0]
genus  = t["taxon"].str.extract(r"g__([^;]+)")[0]
family = t["taxon"].str.extract(r"f__([^;]+)")[0]
unannotated = t["primary_lifestyle"] == ""

# new column that documents every change
t["curation"] = ""
t.loc[t[id_col] == "#q2:types", "curation"] = "categorical"   # column type for QIIME

# (A)
for g, (old, *values) in genus_map.items():
    m = unannotated & (genus == g)
    t.loc[m, cols] = values
    t.loc[m, "curation"] = f"genus not in FungalTraits v1.2; traits of former genus {old}"

# (B)
m = unannotated & (family == "Dipodascaceae") & \
    (genus.isna() | genus.str.contains("Incertae_sedis|unidentified", na=False))
t.loc[m, "growth_form_template"] = "yeast"
note = "Dipodascaceae without genus; growth form yeast (all genera of the family)"
if assign_dipodascaceae_lifestyle:
    t.loc[m, "primary_lifestyle"] = "nectar/tap_saprotroph"
    note += "; lifestyle nectar/tap_saprotroph (5 of 6 genera)"
t.loc[m, "curation"] = note

t.to_csv(traits_curated, sep="\t", index=False)
print("Saved:", traits_curated)

# overview: what was changed, and how many reads it affects
changed = t[(t["curation"] != "") & ~t[id_col].str.startswith("#")].set_index(id_col)
changed["reads"] = rare.reindex(changed.index).sum(axis=1)
changed.groupby("curation").agg(ASVs=("reads", "size"), reads=("reads", "sum"))

Saved: /home/jovyan/FunGut/data/processed_ignored/Fungal_Annotation/02_curation/traits_curated.tsv


,ASVs,reads
curation,,
Dipodascaceae without genus; growth form yeast (all genera of the family),7,54465.0
genus not in FungalTraits v1.2; traits of former genus Candida,5,18198.0
genus not in FungalTraits v1.2; traits of former genus Kazachstania,1,9.0
genus not in FungalTraits v1.2; traits of former genus Mortierella,3,52.0
genus not in FungalTraits v1.2; traits of former genus Psathyrella,1,12.0


#### Now we check if only the intended rows were changed:

In [15]:
orig = pd.read_csv(traits_tsv, sep="\t", dtype=str, keep_default_na=False)
cur  = pd.read_csv(traits_curated, sep="\t", dtype=str, keep_default_na=False)
id_col = cur.columns[0]

diff = orig.ne(cur[orig.columns])           # compares every cell in the original columns
print("Rows:", len(orig), "->", len(cur))
print("New columns:", [c for c in cur.columns if c not in orig.columns])
print("Changed rows:", diff.any(axis=1).sum())
print("Changed columns:", list(diff.columns[diff.any()]))
print("Rows with a curation note:", (cur["curation"].ne("") & ~cur[id_col].str.startswith("#")).sum())

Rows: 682 -> 682
New columns: ['curation']
Changed rows: 17
Changed columns: ['primary_lifestyle', 'secondary_lifestyle', 'growth_form_template', 'fruitbody_type_template']
Rows with a curation note: 17


#### Save visualization:

In [16]:
!qiime metadata tabulate --m-input-file "$traits_curated" --o-visualization "$cur_vis/traits_curated.qzv"

Saved Visualization to: /home/jovyan/FunGut/data/processed/Fungal_Annotation/02_curation/traits_curated.qzv


#### Compare new trait table (curated) with old/original one:

In [17]:
tr_before = load_traits(traits_tsv)
tr_after  = load_traits(traits_curated)

def unannotated_share(tr, col):
    """% of reads per sample from ASVs without an entry in column col"""
    ids = tr.index[tr[col].isna()]
    return rare.loc[rare.index.isin(ids)].sum() / rare.sum() * 100

rows = {}
for col in ["primary_lifestyle", "growth_form_template"]:
    for label, tr in [("before", tr_before), ("after", tr_after)]:
        s = unannotated_share(tr, col)
        rows[(col, label)] = {
            "unannotated ASVs in table": int(rare.index.isin(tr.index[tr[col].isna()]).sum()),
            "mean %": round(s.mean(), 2),
            "75th percentile %": round(s.quantile(0.75), 2),
            "max %": round(s.max(), 2),
            "samples >50%": int((s > 50).sum()),
        }
pd.DataFrame(rows).T

unannotated ASVs in table  mean %  \
primary_lifestyle    before                       78.0    3.06   
                     after                        69.0    2.30   
growth_form_template before                       78.0    3.06   
                     after                        62.0    0.03   

                             75th percentile %  max %  samples >50%  
primary_lifestyle    before               0.12  98.19           3.0  
                     after                0.07  70.96           2.0  
growth_form_template before               0.12  98.19           3.0  
                     after                0.00   1.05           0.0

#### Lets look at the 2 samples we have to exclude from the analysis (because they have over 50% unannotated reads)

In [18]:
share_life = unannotated_share(tr_after, "primary_lifestyle")
exclude = share_life[share_life > 50].sort_values(ascending=False)
print(len(exclude), "samples with >50% reads without a lifestyle:\n")

unann_ids = tr_after.index[tr_after["primary_lifestyle"].isna()]
for s in exclude.index:
    top = rare.loc[rare.index.isin(unann_ids), s].sort_values(ascending=False).head(1)
    print(f"{s}: {exclude[s]:.1f}% unannotated | dominant ASV: {int(top.iloc[0])} reads, "
          f"{tr_after.loc[top.index[0], 'taxon'].split(';')[-1]}")

if len(exclude) > 0:
    pd.DataFrame(index=pd.Index(exclude.index, name="sample-id")).to_csv(exclude_tsv, sep="\t")
    print("\nSaved:", exclude_tsv)

2 samples with >50% reads without a lifestyle:

ERR5327488: 71.0% unannotated | dominant ASV: 11353 reads, s__Dipodascaceae_sp
ERR5327599: 60.9% unannotated | dominant ASV: 9750 reads, f__Dipodascaceae

Saved: /home/jovyan/FunGut/data/processed_ignored/Fungal_Annotation/02_curation/exclude_samples_lifestyle.tsv


In [19]:
meta = load_metadata(metadata)
excl = pd.read_csv(exclude_tsv, sep="\t", index_col=0).index.astype(str)

show_cols = cat_main + num_main + cat_extra + num_extra
out = meta.loc[meta.index.isin(excl), show_cols].copy()
out["% reads without lifestyle"] = share_life.reindex(out.index).round(1)
out

,diet_type_sample,gluten_sample,ibd_sample,Settlement_Type,bmi_sample,sex_sample,country_sample,age_years_sample,% reads without lifestyle
sample-id,,,,,,,,,
ERR5327488,NaN,NaN,NaN,NaN,NaN,NaN,Germany,NaN,71.0
ERR5327599,Omnivore,No,I do not have this condition,Rural,21.97,female,Isle of Man,58.0,60.9


-> Thats interesting: One of the samples has NaN in every column except country and the other one is the one sample from Isle of Man (and in the other categories lies in the majority -> omnivore, No gluten intolerance and no IBD). So we exclude those samples for this analysis.

# 02 Remove the non-intrinsic fungi